# Trabalho Prático – Obtenção e Visualização de Dados com Python

**Integrantes:**
- Luan Nascimento Caetano – CP3044696
- Rafael da Silva Oliveira – CP3044564

## Tema escolhido
*(Responsável: Rafael)*

O tema escolhido é o **basquete brasileiro**, a partir das estatísticas dos jogadores do **NBB (Novo Basquete Brasil)**, principal campeonato de basquete masculino do país, organizado pela Liga Nacional de Basquete (LNB).

O conjunto de dados reúne o desempenho individual de cada jogador em cada temporada, de **2008-09 a 2021-22**. Por ter muitos registros e muitas colunas numéricas, ele permite praticar todas as etapas pedidas no trabalho: **organizar → selecionar → limpar → visualizar**.

## Origem e significado dos dados
*(Responsável: Rafael)*

**Fonte:** [Novo Basquete Brasil (NBB) Stats - All Seasons](https://www.kaggle.com/datasets/gabrielpastorello/novo-basquete-brasil-nbb-stats-all-seasons), publicado no Kaggle por Gabriel Pastorello, com licença CC0 (domínio público). Os dados foram reunidos a partir das estatísticas oficiais do NBB.

**Arquivos (pasta `dados/`):**

| Arquivo | Conteúdo |
|---|---|
| `Base_NBB.csv` | Arquivo principal: 3.201 registros (um por jogador em cada temporada) e 70 colunas |
| `Descricao_Variaveis.xlsx` | Significado de cada sigla usada nas colunas |
| `Classificacao.csv` | Classificação das equipes na fase regular de cada temporada |

**Como as colunas estão organizadas:** cada estatística aparece duas vezes, com os sufixos `_TOTAL` (soma na temporada) e `_PORJOGO` (média por partida). Exemplo: `PTS_TOTAL` e `PTS_PORJOGO`.

**Principais colunas utilizadas:**

| Coluna | Significado |
|---|---|
| `Jogador` | Nome do jogador |
| `Equipe` | Time do jogador na temporada |
| `Temporada` | Temporada do campeonato (ex.: 2021-22) |
| `JO_TOTAL` | Número de partidas jogadas |
| `PTS_PORJOGO` | Pontos por jogo |
| `RT_PORJOGO` | Rebotes por jogo |
| `AS_PORJOGO` | Assistências por jogo |
| `EF_PORJOGO` | Eficiência por jogo: índice que soma as ações positivas (pontos, rebotes, assistências…) e desconta as negativas (erros, arremessos perdidos…) |
| `MVP` | 1 se o jogador foi eleito o melhor da temporada, 0 caso contrário |

As demais siglas estão explicadas no arquivo `Descricao_Variaveis.xlsx`.

In [ ]:
# ============================================================
# Bibliotecas utilizadas no trabalho
# Rode esta célula primeiro: as outras dependem dela.
# ============================================================

import csv                       # módulo nativo do Python para ler arquivos CSV (Etapa 1)
import numpy as np               # arrays e cálculos numéricos (Etapa 2)
import pandas as pd              # DataFrames: seleção, filtragem e limpeza (Etapa 3)
import matplotlib.pyplot as plt  # gráficos (Etapa 4)

# Caminho do arquivo de dados, relativo à pasta do notebook.
# Fica em uma variável para ser usado em todas as etapas.
CAMINHO_DADOS = "dados/Base_NBB.csv"

## Etapa 1 – Organização inicial dos dados (estruturas nativas do Python)
*(Responsável: Rafael)*

**O que o enunciado pede (item 1 – Organização inicial dos dados):** representar ou manipular os dados com pelo menos uma estrutura de dados nativa do Python (lista, lista de listas, tupla, dicionário ou lista de dicionários). Parte desses dados deve depois ser convertida em estruturas do NumPy ou do Pandas.

**Requisito obrigatório (Python):** uso de estruturas de dados nativas, condicionais e/ou repetições.

Nesta etapa o arquivo é lido **sem NumPy e sem Pandas**, apenas com o módulo `csv` da biblioteca padrão do Python. Assim, toda a organização inicial é feita com estruturas nativas:

| Estrutura | Variável | Passo |
|---|---|---|
| Tupla | `COLUNAS_ETAPA1` | 1.1 |
| Lista de dicionários | `jogadores` | 1.1 |
| Dicionário | `registros_por_temporada` | 1.3 |
| Lista (de tuplas) | `destaques` | 1.4 |
| Lista de listas | `estatisticas` | 1.5 (convertida em array na Etapa 2) |

### 1.1 Leitura do arquivo em uma lista de dicionários

- As colunas escolhidas ficam em uma **tupla**, porque essa lista não muda durante o programa.
- O `csv.DictReader` lê o arquivo linha por linha e entrega cada linha como um dicionário `{coluna: valor}`.
- Um laço `for` percorre as linhas e monta um dicionário só com as colunas escolhidas, convertendo os números de texto para `int` ou `float`.
- Cada dicionário é adicionado à lista `jogadores`, formando uma **lista de dicionários**.

In [ ]:
# ------------------------------------------------------------
# 1.1 Leitura do CSV em uma LISTA DE DICIONÁRIOS
# ------------------------------------------------------------

# TUPLA com os nomes das colunas usadas nesta etapa.
# Usamos tupla (e não lista) porque esses nomes não vão mudar.
COLUNAS_ETAPA1 = ("Jogador", "Equipe", "Temporada", "JO_TOTAL",
                  "PTS_PORJOGO", "RT_PORJOGO", "AS_PORJOGO", "EF_PORJOGO", "MVP")

# LISTA vazia que vai receber um dicionário por jogador/temporada
jogadores = []

# "with open" abre o arquivo e fecha automaticamente ao terminar o bloco
with open(CAMINHO_DADOS, encoding="utf-8") as arquivo:
    # DictReader usa a primeira linha do CSV (cabeçalho) como chaves
    # e entrega cada linha seguinte como um dicionário {coluna: valor}
    leitor = csv.DictReader(arquivo)

    # REPETIÇÃO: percorre o arquivo linha por linha
    for linha in leitor:
        # O CSV guarda tudo como texto (str), então convertemos os números:
        #   int()   -> números inteiros (quantidade de jogos)
        #   float() -> números com casas decimais (médias por jogo)
        registro = {
            "Jogador": linha["Jogador"],                      # texto
            "Equipe": linha["Equipe"],                        # texto
            "Temporada": linha["Temporada"],                  # texto, ex.: "2021-22"
            "JO_TOTAL": int(linha["JO_TOTAL"]),               # partidas jogadas
            "PTS_PORJOGO": float(linha["PTS_PORJOGO"]),       # pontos por jogo
            "RT_PORJOGO": float(linha["RT_PORJOGO"]),         # rebotes por jogo
            "AS_PORJOGO": float(linha["AS_PORJOGO"]),         # assistências por jogo
            "EF_PORJOGO": float(linha["EF_PORJOGO"]),         # eficiência por jogo
            "MVP": int(float(linha["MVP"])),                  # vem como "0.0" ou "1.0" -> 0 ou 1
        }

        # Adiciona o dicionário ao final da lista
        jogadores.append(registro)

# Confere o resultado: quantos registros e qual estrutura foi criada
print(f"Registros lidos: {len(jogadores)}")
print(f"Tipo da estrutura: {type(jogadores).__name__} de {type(jogadores[0]).__name__}")

### 1.2 Visão geral dos registros

Um laço percorre os três primeiros registros (`jogadores[:3]`) e, dentro dele, outro laço percorre a tupla `COLUNAS_ETAPA1` para mostrar cada campo.

In [ ]:
# ------------------------------------------------------------
# 1.2 Visão geral dos registros
# ------------------------------------------------------------

# len() devolve a quantidade de elementos da lista e da tupla
print(f"Total de registros: {len(jogadores)}")
print(f"Colunas selecionadas: {len(COLUNAS_ETAPA1)}\n")

# jogadores[:3] pega os 3 primeiros elementos da lista (fatiamento)
for registro in jogadores[:3]:
    # Laço interno: percorre a tupla de colunas para mostrar cada campo
    for coluna in COLUNAS_ETAPA1:
        # {coluna:>12} alinha o nome da coluna à direita em 12 espaços
        print(f"{coluna:>12}: {registro[coluna]}")
    print("-" * 30)  # linha separadora entre um jogador e outro

### 1.3 Dicionário: quantidade de jogadores por temporada

Um **dicionário** guarda a contagem de registros de cada temporada (chave = temporada, valor = quantidade). A **condicional** `if` verifica se a temporada já está no dicionário: se estiver, soma 1; se não, cria a chave com valor 1.

In [ ]:
# ------------------------------------------------------------
# 1.3 DICIONÁRIO com a quantidade de jogadores por temporada
# ------------------------------------------------------------

# Dicionário vazio: a chave será a temporada e o valor, a contagem
registros_por_temporada = {}

# REPETIÇÃO: passa por todos os registros
for registro in jogadores:
    temporada = registro["Temporada"]

    # CONDICIONAL: a temporada já apareceu antes?
    if temporada in registros_por_temporada:
        registros_por_temporada[temporada] += 1   # sim: soma 1 à contagem
    else:
        registros_por_temporada[temporada] = 1    # não: cria a chave com 1

# len() de um dicionário devolve a quantidade de chaves
print(f"Temporadas no conjunto: {len(registros_por_temporada)}\n")

# sorted() percorre as chaves em ordem (da temporada mais antiga à mais recente)
for temporada in sorted(registros_por_temporada):
    print(f"{temporada}: {registros_por_temporada[temporada]} jogadores")

### 1.4 Condicionais: destaques e MVPs

- **Destaques:** jogadores com pelo menos 15 pontos por jogo **e** pelo menos 10 partidas jogadas (para não contar quem jogou pouquíssimos jogos). Cada destaque é guardado como uma **tupla** dentro da lista `destaques`.
- **MVPs:** jogadores com `MVP == 1`, o melhor de cada temporada.

In [ ]:
# ------------------------------------------------------------
# 1.4 CONDICIONAIS: destaques e MVPs
# ------------------------------------------------------------

# Critérios do filtro, em variáveis para facilitar a mudança
MIN_PONTOS = 15   # mínimo de pontos por jogo
MIN_JOGOS = 10    # mínimo de partidas (evita quem jogou só 1 ou 2 jogos)

# LISTA que vai guardar uma TUPLA para cada destaque
destaques = []

for registro in jogadores:
    # CONDICIONAL com "and": as duas condições precisam ser verdadeiras
    if registro["PTS_PORJOGO"] >= MIN_PONTOS and registro["JO_TOTAL"] >= MIN_JOGOS:
        # Tupla (jogador, equipe, temporada, pontos): dados fixos do destaque
        destaques.append((registro["Jogador"], registro["Equipe"],
                          registro["Temporada"], registro["PTS_PORJOGO"]))

print(f"Jogadores com {MIN_PONTOS}+ pontos por jogo e {MIN_JOGOS}+ partidas: {len(destaques)}\n")

# Mostra os 5 primeiros destaques.
# "jogador, equipe, temporada, pontos" desempacota cada tupla em 4 variáveis
for jogador, equipe, temporada, pontos in destaques[:5]:
    print(f"{temporada} | {jogador} ({equipe}): {pontos} pontos por jogo")

# MVP: a coluna vale 1 apenas para o melhor jogador de cada temporada
print("\nMVPs de cada temporada:")
for registro in jogadores:
    if registro["MVP"] == 1:
        print(f"{registro['Temporada']} | {registro['Jogador']} ({registro['Equipe']})")

### 1.5 Lista de listas: preparação para o NumPy

O enunciado pede que parte dos dados organizados aqui seja convertida depois em estruturas do NumPy ou do Pandas. Por isso, montamos uma **lista de listas** só com valores numéricos: cada linha é um registro e cada coluna é uma estatística por jogo (pontos, rebotes, assistências e eficiência). Essa lista será convertida em array NumPy na Etapa 2.

In [ ]:
# ------------------------------------------------------------
# 1.5 LISTA DE LISTAS: preparação para o NumPy (Etapa 2)
# ------------------------------------------------------------

# Tupla com o nome de cada coluna da lista de listas, na mesma ordem.
# Será usada na Etapa 2 para identificar as colunas do array.
NOMES_ESTATISTICAS = ("Pontos", "Rebotes", "Assistências", "Eficiência")

# Lista de listas: cada item é uma lista com 4 números de um registro
estatisticas = []

for registro in jogadores:
    # Monta uma linha só com valores numéricos, na ordem de NOMES_ESTATISTICAS
    linha = [registro["PTS_PORJOGO"], registro["RT_PORJOGO"],
             registro["AS_PORJOGO"], registro["EF_PORJOGO"]]
    estatisticas.append(linha)

# Dimensão da "tabela": número de linhas x número de colunas
print(f"Dimensão: {len(estatisticas)} linhas x {len(estatisticas[0])} colunas")
print(f"Colunas: {NOMES_ESTATISTICAS}")
print(f"Três primeiras linhas: {estatisticas[:3]}")

## Etapa 2 – Utilização do NumPy
*(Responsável: Rafael)*

In [ ]:
# arrays, indexação/slicing e pelo menos duas operações numéricas

## Etapa 3 – Utilização do Pandas
*(Responsável: Luan)*

### Etapas de tratamento dos dados
explicar quais dados inválidos foram encontrados e como foram tratados.

In [ ]:
# criação/importação do DataFrame, seleção de linhas e colunas

In [ ]:
# identificação e tratamento/remoção de dados ausentes ou inválidos

In [ ]:
# filtragem por condição e ordenação

In [ ]:
# pelo menos duas estatísticas ou agregações

## Etapa 4 – Visualização com Matplotlib

### Gráfico 1
*(Responsável: Rafael)*

In [ ]:
# gráfico 1 (título, eixos e legenda)

### Gráfico 2
*(Responsável: Rafael)*

In [ ]:
# gráfico 2 (título, eixos e legenda)

### Gráfico 3
*(Responsável: Luan)*

In [ ]:
# gráfico 3 – tipo diferente dos gráficos 1 e 2